In [ ]:
%load_ext autoreload 
%autoreload 2
%matplotlib inline
import sys
import cProfile #for checking the nr of calls and execution time
import pstats
from pstats import SortKey
from scipy.stats import gaussian_kde
import json
import os
import re
import numpy as np
import multiple_planets_gas_acc as code_gas
from functions_pebble_accretion import *
from functions import *
import functions_plotting as plot
import sim_loader as sim_load
import functions_plotting as plot
import matplotlib.pyplot as plt
import matplotlib as mpl
import astropy.units as u
import pandas as pd
from matplotlib.ticker import ScalarFormatter, LogFormatter, LogLocator, MultipleLocator, AutoMinorLocator
from matplotlib import cm, ticker
from matplotlib import colors
import matplotlib.gridspec as gridspec
import matplotlib.patches as patch
from matplotlib.offsetbox import AnchoredText
from matplotlib.patches import Patch
import matplotlib.lines as mlines 
from matplotlib.animation import FuncAnimation
from matplotlib import pyplot as plt, ticker as mticker
import sim_loader as sim_load
import h5py
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
    "axes.labelsize": 15,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.direction": "in",
    "ytick.direction": "in",
    # major ticks
    "xtick.major.size": 12,
    "ytick.major.size": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    # minor ticks
    "xtick.minor.size": 8,
    "ytick.minor.size": 8,
    "xtick.minor.visible": True,   # minor ticks are hidden by default — need this to actually see them
    "ytick.minor.visible": True,

})

In [ ]:

# ============================================================
# FILE SORTING
# ============================================================

def extract_initial_time(filename):

    with h5py.File(filename, 'r') as f:

        time = f['simulation']['time'][()]

        if np.ndim(time) > 0:
            return float(time[0])

        return float(time)




In [ ]:
# ============================================================
# LOAD SIMULATIONS 
# ============================================================
folder_path_multi = ["sims/opacities/1Msun/Mdot_scatter/multiples/kappa_0005","sims/opacities/1Msun/Mdot_scatter/multiples/kappa_1"]
folder_path_single = ["sims/opacities/1Msun/Mdot_scatter/singles/kappa_0005","sims/opacities/1Msun/Mdot_scatter/singles/kappa_1"]

simulations_multi = []
sim_parameters_multi = []
parameters_multi = []
simulations_single = []
sim_parameters_single = []
parameters_single = []

for folder in folder_path_multi:
    h5_files = sorted(
        [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".h5")],
        key=extract_initial_time
    )
    simulations_multi.append([sim_load.HDF5toSimRes(f) for f in h5_files])
    sim_parameters_multi.append([sim_load.load_sim_params(f) for f in h5_files])
    parameters_multi.append([sim_load.load_params(f) for f in h5_files])

for folder in folder_path_single:
    h5_files = sorted(
        [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".h5")],
        key=extract_initial_time
    )
    simulations_single.append([sim_load.HDF5toSimRes(f) for f in h5_files])
    sim_parameters_single.append([sim_load.load_sim_params(f) for f in h5_files])
    parameters_single.append([sim_load.load_params(f) for f in h5_files])

In [ ]:
alpha_transp = 0.2
c = ['orange','teal']

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

for i, folder in enumerate(folder_path_multi):
    for sim, params, sim_params in zip(simulations_multi[i], parameters_multi[i], sim_parameters_multi[i]):
        for p in range(sim_params.nr_planets):
            idx = plot.idxs(sim.time.value, sim.mass[p].value, sim.position[p].value, sim.filter_fraction[p],
                             sim.dR_dt[p].value, sim.dM_dt[p].value, params, True)
            axs[0].scatter(sim.position[p, -1], sim.mass[p, -1], color=c[i])

for i, folder in enumerate(folder_path_single):
    for sim, params, sim_params in zip(simulations_single[i], parameters_single[i], sim_parameters_single[i]):
        for p in range(sim_params.nr_planets):
            idx = plot.idxs(sim.time.value, sim.mass[p].value, sim.position[p].value, sim.filter_fraction[p],
                             sim.dR_dt[p].value, sim.dM_dt[p].value, params, True)
            axs[1].scatter(sim.position[p, -1], sim.mass[p, -1], color=c[i])

## aesthetics of the plot
a_p0 = np.geomspace(0.01, 100, 100)
for i in range(2):
    axs[i].axvline(r_magnetic_cavity(sim_params.t_in, params), linestyle = '-.', color = 'grey', alpha = 0.1)
    axs[i].axvline(r_magnetic_cavity(sim_params.t_fin, params), linestyle = '-.', color = 'grey', alpha = 0.1)
    axs[i].axvspan(r_magnetic_cavity(sim_params.t_in, params), r_magnetic_cavity(sim_params.t_fin, params),facecolor='none', hatch='/', edgecolor='gray', alpha =alpha_transp)
    axs[i].loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
    axs[i].loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
    axs[i].fill_between(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params),  color='slateblue', alpha=0.1)   
    plot.boxes(axs[i], 1e-2)
    axs[i].set_xlabel('r [AU]', fontsize=25, labelpad=20)
    axs[i].set_xlim(0.01, 1e2)
    axs[i].set_ylim(1e-3, 1e4)
    axs[i].tick_params(axis="both", which="major", direction='in', size=15, labelsize=18)
    axs[i].tick_params(axis="both", which="minor", direction='in', size=10)
    axs[i].yaxis.set_major_locator(mticker.LogLocator(numticks=999))
    axs[i].yaxis.set_minor_locator(mticker.LogLocator(numticks=999, subs="auto"))
    plot.all_x_ticks(axs[i], num_ticks=100)
    axs[i].set_xscale('log')
    axs[i].set_yscale('log')
    axs[i].axhline(13*u.M_jup.to(u.M_earth), color = 'grey', linestyle = '--', alpha = 0.5)
kappa_values = [0.005, 1.0]  # List of kappa values corresponding to the folders
# --- legend: one color per kappa value ---
legend_handles = [
    mlines.Line2D([], [], color=c[i], marker='o', linestyle='None', markersize=8,
                  label=fr'$\kappa$ = {kappa_values[i]}m$^2$/kg')
    for i in range(len(folder_path_multi))
]
axs[0].legend(handles=legend_handles, fontsize=14, loc='best')
axs[0].set_ylabel('M [$M_{\oplus}$]', fontsize=25, labelpad=20)

axs[0].set_title("Multiplanet systems, $N_{\\rm planets} = 5$", fontsize=15, pad=20)
axs[1].set_title("Single planet systems", fontsize=15, pad=20)
plt.savefig("figures/opacities_scatter_multiplanet.png", bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def classify_planets(position, mass):
    if mass > 1 and mass < 20 and position < 1:
        return "Super-Earth"
    elif mass > 30 and mass < 3000:
        if position > 1:
            return "Cold Giant"
        elif position < 1 and position > 0.1:
            return "Warm Giant"
        else:
            return "Hot Giant"
    else:
        return None

categories = ["Super-Earth", "Warm Giant", "Cold Giant", "Hot Giant"]

def category_counts(simulations, sim_parameters):
    classes = [c for sim in simulations for p in range(sim_parameters[0].nr_planets)
               if (c := classify_planets(sim.position[p, -1].value, sim.mass[p, -1].value)) is not None]
    return [classes.count(cat) for cat in categories]

kappa_values = ["1 $\\rm m^2$/kg", "0.005 $\\rm m^2/kg$"] 
colors = ['teal', 'darkorange']  # one color per kappa

x = np.arange(len(categories))
n_bars = 2 * len(kappa_values)  # multi + single, per kappa
width = 0.8 / n_bars

fig, ax = plt.subplots(1, 1, figsize=(9, 6))

bar_idx = 0
for i, kappa in enumerate(kappa_values):
    # multiplanet: solid fill
    offset = (bar_idx - (n_bars - 1) / 2) * width
    vals = category_counts(simulations_multi[i], sim_parameters_multi[i])
    ax.bar(x + offset, vals, width, color=colors[i], alpha=0.8,
           label=f"Multi, κ = {kappa}")
    bar_idx += 1

    # single planet: hatched
    offset = (bar_idx - (n_bars - 1) / 2) * width
    vals = category_counts(simulations_single[i], sim_parameters_single[i])
    ax.bar(x + offset, vals, width, color=colors[i], alpha=0.8,
           hatch='//', edgecolor='black', label=f"Single, κ = {kappa}")
    bar_idx += 1

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("Number of planets", fontsize=15)
ax.set_xlabel("Planet Type", fontsize=15)
ax.set_title("Planet types", fontsize=14)
ax.legend()
plt.tight_layout()
plt.savefig("figures/opacities/planet_type_hist.png", bbox_inches='tight')

## Scaled down accretion rates

In [ ]:
# ============================================================
# LOAD SIMULATIONS 
# ============================================================
folder_path_single = ["sims/opacities/1Msun/Mdot_scatter/Mdot_downscale_01/migration_downscale_05/singles/kappa_0005","sims/opacities/1Msun/Mdot_scatter/Mdot_downscale_01/migration_downscale_05/singles/kappa_1"]


simulations_single = []
sim_parameters_single = []
parameters_single = []

for folder in folder_path_single:
    h5_files = sorted(
        [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".h5")],
        key=extract_initial_time
    )
    simulations_single.append([sim_load.HDF5toSimRes(f) for f in h5_files])
    sim_parameters_single.append([sim_load.load_sim_params(f) for f in h5_files])
    parameters_single.append([sim_load.load_params(f) for f in h5_files])

In [ ]:
alpha_transp = 0.2
c = ['orange','teal']

fig, axs = plt.subplots(1, 1, figsize=(6, 6))

for i, folder in enumerate(folder_path_single):
    for sim, params, sim_params in zip(simulations_single[i], parameters_single[i], sim_parameters_single[i]):
        for p in range(sim_params.nr_planets):
            idx = plot.idxs(sim.time.value, sim.mass[p].value, sim.position[p].value, sim.filter_fraction[p],
                             sim.dR_dt[p].value, sim.dM_dt[p].value, params, True)
            axs.scatter(sim.position[p, -1], sim.mass[p, -1], color=c[i])

## aesthetics of the plot
a_p0 = np.geomspace(0.01, 100, 100)
axs.loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
axs.loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
axs.fill_between(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params),  color='slateblue', alpha=0.1)   

axs.set_xlabel('r [AU]', fontsize=25, labelpad=20)
axs.set_xlim(0.01, 1e2)
axs.set_ylim(1e-3, 1e4)
axs.tick_params(axis="both", which="major", direction='in', size=15, labelsize=18)
axs.tick_params(axis="both", which="minor", direction='in', size=10)
axs.yaxis.set_major_locator(mticker.LogLocator(numticks=999))
axs.yaxis.set_minor_locator(mticker.LogLocator(numticks=999, subs="auto"))
plot.all_x_ticks(axs, num_ticks=100)
axs.set_xscale('log')
axs.set_yscale('log')
axs.axhline(13*u.M_jup.to(u.M_earth), color = 'grey', linestyle = '--', alpha = 0.5)
kappa_values = [0.005, 1.0]  # List of kappa values corresponding to the folders
# --- legend: one color per kappa value ---
legend_handles = [
    mlines.Line2D([], [], color=c[i], marker='o', linestyle='None', markersize=8,
                  label=fr'$\kappa$ = {kappa_values[i]}m$^2$/kg')
    for i in range(len(folder_path_single))
]
axs.legend(handles=legend_handles, fontsize=14, loc='best')
axs.set_ylabel('M [$M_{\oplus}$]', fontsize=25, labelpad=20)
axs.set_title("Singles, $\dot{M}_{\star} = 0.1 \dot{M}_{\star, 0}$, 0.5 times migration", fontsize=15, pad=20)
plt.savefig("figures/opacities/opacities_scatter_scaledown_Mdot_01_migration_downscale_05.png", bbox_inches='tight')
plt.show()

In [ ]:
# color scheme definition
kept_types = ['SE', 'WG', 'CG', 'HJ']

# restrict sampling range to avoid the bright yellow near the top of inferno
cmap_min, cmap_max = 0.0, 0.85

color_map = {
    ptype: plt.cm.inferno(cmap_min + i / (len(kept_types) - 1) * (cmap_max - cmap_min))
    for i, ptype in enumerate(kept_types)
}
sim_list_idx = 1  # index of the simulation to analyze
kappa_value = 0.005 if sim_list_idx == 0 else 1.0  # index of the kappa value to analyze (0 for 0.005, 1 for 1.0)
sim, params, sim_params = simulations_single[sim_list_idx], parameters_single[sim_list_idx], sim_parameters_single[sim_list_idx]
planet_records = planet_type_counter(sim, params, sim_params, per_planet=True)
df = pd.DataFrame(planet_records)
fig, ax = plt.subplots(1, 1, figsize=(9, 6))

bins=30
log_bins_range=(-2, 2)
# which planets to plot
types = [t for t in df['type'].unique() if t not in ['unclassified', 'sub_E', 'sub_giants_in', 'sub_giants_out', 'giant_other', 'terr_in', 'SE_sabotta']]
bin_edges = (np.logspace(*log_bins_range, bins))
total_planets = len(df)
for ptype in types:
    subset = df.loc[df['type'] == ptype, 'position_au']
    if not subset.empty:
        ax.hist(subset, bins=bin_edges, histtype='step', linewidth=2, label=ptype, color=color_map[ptype])

ax.set_xscale('log')
ax.set_xlabel('$a_{\\rm p}$ [AU]')
ax.set_ylabel('Occurrence [count]')
ax.set_title('$\dot{M}_{\star} = 0.1 \dot{M}_{\star, 0}$, 0.5 times migration, $\kappa = ' + str(kappa_value) + ' \\rm m^2/kg$ ')
ax.legend()
plt.savefig("figures/opacities/planet_occurrence_vs_position_hist_01Mdot_05mig_" + str(kappa_value) + "_kappa.png", bbox_inches='tight')

fig, ax = plt.subplots(1, 1, figsize=(9, 6))

bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])  # geometric mean, correct for log bins

for ptype in types:
    subset = df.loc[df['type'] == ptype, 'position_au']
    if not subset.empty:
        counts, _ = np.histogram(subset, bins=bin_edges)
        ax.plot(bin_centers, counts, marker='o', linewidth=2, label=ptype, color=color_map[ptype])

ax.set_xscale('log')
ax.set_xlabel('$a_{\\rm p}$ [AU]')
ax.set_ylabel('Occurrence [count]')
ax.set_title('$\dot{M}_{\star} = 0.1 \dot{M}_{\star, 0}$, 0.5 times migration, $\kappa = ' + str(kappa_value) + ' \\rm m^2/kg$ ')
ax.legend()
plt.savefig("figures/opacities/planet_occurrence_vs_position_01Mdot_05mig_" + str(kappa_value) + "_kappa.png", bbox_inches='tight')

fig, ax = plt.subplots(1, 1, figsize=(9, 6))
for ptype in types:
    subset = df.loc[df['type'] == ptype]
    if not subset.empty:
        ax.scatter(subset['position_au'], subset['mass_earth'], s=20, alpha=0.7, label=ptype, color=color_map[ptype])

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('$a_{\\rm p}$ [AU]')
ax.set_ylabel('$M_{\\rm p}$ [$M_\\oplus$]')
ax.set_title('$\dot{M}_{\star} = 0.1 \dot{M}_{\star, 0}$, 0.5 times migration, $\kappa = ' + str(kappa_value) + ' \\rm m^2/kg$ ')
ax.legend()
plt.savefig("figures/opacities/planet_mass_vs_position_scatter_01Mdot_05mig_" + str(kappa_value) + "_kappa.png", bbox_inches='tight')
print('HJ cmap',color_map['HJ'])
df

## Parameter exploration with joint plots
plot the super-earth and cold giant occurrence rates as a function of the Mdot scaling and varying alpha disc. Color coding for different planet types, markers and linestyle for different opacities

In [ ]:
# ============================================================
# MDOT SCALING ANALYSIS
# ============================================================
folder_path = ["sims/opacities/1Msun/Mdot_scatter/Mdot_downscale_01/singles/kappa_0005",
               "sims/opacities/1Msun/Mdot_scatter/Mdot_downscale_04/singles/kappa_0005",
               "sims/opacities/1Msun/Mdot_scatter/Mdot_downscale_06/singles/kappa_0005",
               "sims/opacities/1Msun/Mdot_scatter/Mdot_downscale_08/singles/kappa_0005",
               "sims/opacities/1Msun/Mdot_scatter/kappa_0005"]


simulations = []
sim_parameters = []
parameters = []

for folder in folder_path:
    h5_files = sorted(
        [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".h5")],
        key=extract_initial_time
    )
    simulations.append([sim_load.HDF5toSimRes(f) for f in h5_files])
    sim_parameters.append([sim_load.load_sim_params(f) for f in h5_files])
    parameters.append([sim_load.load_params(f) for f in h5_files])

In [ ]:
data_summary = plot.create_joint_plot_dataset(simulations, parameters, sim_parameters, per_planet=True)
data_summary

In [ ]:
plot.plot_joint_distribution_Mdot(data_summary, filename="planet_position_mass_joint_kde_Mdot_scaling_kappa0005_per_curve", 
            title='Planet populations for different $\dot{M}_{\star}$ scaling factors, $\kappa_{\mathrm{env}} = 0.005 \mathrm{m}^2/\mathrm{kg}$',
            normalise_per_curve=False)

In [ ]:
# ============================================================
# MDOT SCALING ANALYSIS
# ============================================================
folder_path = ["sims/opacities/1Msun/Mdot_scatter/Mdot_downscale_01/singles/kappa_1",
               "sims/opacities/1Msun/Mdot_scatter/Mdot_downscale_04/singles/kappa_1",
               "sims/opacities/1Msun/Mdot_scatter/Mdot_downscale_06/singles/kappa_1",
               "sims/opacities/1Msun/Mdot_scatter/Mdot_downscale_08/singles/kappa_1",]


simulations = []
sim_parameters = []
parameters = []

for folder in folder_path:
    h5_files = sorted(
        [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".h5")],
        key=extract_initial_time
    )
    simulations.append([sim_load.HDF5toSimRes(f) for f in h5_files])
    sim_parameters.append([sim_load.load_sim_params(f) for f in h5_files])
    parameters.append([sim_load.load_params(f) for f in h5_files])

In [ ]:
data_summary = plot.create_joint_plot_dataset(simulations, parameters, sim_parameters, per_planet=True)
data_summary

In [ ]:
plot.plot_joint_distribution_Mdot(data_summary, filename="planet_position_mass_joint_kde_Mdot_scaling_kappa1", 
            title=r'Planet populations for different $\dot{M}_{\star}$ scaling factors, $\kappa_{\mathrm{env}} = 1 \mathrm{m}^2/\mathrm{kg}$',
            normalise_per_curve=False)

In [ ]:
# ============================================================
# MDOT SCALING ANALYSIS
# ============================================================
folder_path = ["sims/opacities/1Msun/Mdot_scatter/different_alphas/alpha_1e-4/kappa_1",
               "sims/opacities/1Msun/Mdot_scatter/different_alphas/alpha_5e-4/kappa_1",
               "sims/opacities/1Msun/Mdot_scatter/different_alphas/alpha_1e-3/kappa_1",
               "sims/opacities/1Msun/Mdot_scatter/different_alphas/alpha_5e-3/kappa_1",
               "sims/opacities/1Msun/Mdot_scatter/different_alphas/alpha_1e-2/kappa_1"]


simulations = []
sim_parameters = []
parameters = []

for folder in folder_path:
    h5_files = sorted(
        [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".h5")],
        key=extract_initial_time
    )
    simulations.append([sim_load.HDF5toSimRes(f) for f in h5_files])
    sim_parameters.append([sim_load.load_sim_params(f) for f in h5_files])
    parameters.append([sim_load.load_params(f) for f in h5_files])

In [ ]:
data_summary = plot.create_joint_plot_dataset(simulations, parameters, sim_parameters, per_planet=True)
plot.plot_joint_distribution(data_summary, filename="planet_position_mass_joint_kde_alpha_scaling_kappa1", 
            title=r'Planet populations for different $\alpha_{\nu}$ scaling factors, $\kappa_{\mathrm{env}} = 1 \mathrm{m}^2/\mathrm{kg}$',
            normalise_per_curve=False, group_by='alpha')

In [ ]:
alpha_transp = 0.2
c = ['orange','teal', 'green', 'purple', 'red']

fig, axs = plt.subplots(1, 1, figsize=(6, 6))

for i, folder in enumerate(folder_path):
    for sim, params, sim_params in zip(simulations[i], parameters[i], sim_parameters[i]):
        for p in range(sim_params.nr_planets):
            idx = plot.idxs(sim.time.value, sim.mass[p].value, sim.position[p].value, sim.filter_fraction[p],
                             sim.dR_dt[p].value, sim.dM_dt[p].value, params, True)
            axs.scatter(sim.position[p, -1], sim.mass[p, -1], color=c[i])

## aesthetics of the plot
a_p0 = np.geomspace(0.01, 100, 100)
axs.axvline(r_magnetic_cavity(sim_params.t_in, params), linestyle = '-.', color = 'grey', alpha = 0.1)
axs.axvline(r_magnetic_cavity(sim_params.t_fin, params), linestyle = '-.', color = 'grey', alpha = 0.1)
axs.axvspan(r_magnetic_cavity(sim_params.t_in, params), r_magnetic_cavity(sim_params.t_fin, params),facecolor='none', hatch='/', edgecolor='gray', alpha =alpha_transp)
axs.loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
axs.loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
axs.fill_between(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params),  color='slateblue', alpha=0.1)   

axs.set_xlabel('r [AU]', fontsize=25, labelpad=20)
axs.set_xlim(0.01, 1e2)
axs.set_ylim(1e-3, 1e4)
axs.tick_params(axis="both", which="major", direction='in', size=15, labelsize=18)
axs.tick_params(axis="both", which="minor", direction='in', size=10)
axs.yaxis.set_major_locator(mticker.LogLocator(numticks=999))
axs.yaxis.set_minor_locator(mticker.LogLocator(numticks=999, subs="auto"))
plot.all_x_ticks(axs, num_ticks=100)
axs.set_xscale('log')
axs.set_yscale('log')
axs.axhline(13*u.M_jup.to(u.M_earth), color = 'grey', linestyle = '--', alpha = 0.5)
kappa_values = [0.005, 1.0]  # List of kappa values corresponding to the folders
# --- legend: one color per kappa value ---
legend_handles = [
    mlines.Line2D([], [], color=c[i], marker='o', linestyle='None', markersize=8,
                  label=fr'$\kappa$ = {kappa_values[i]}m$^2$/kg')
    for i in range(len(folder_path))
]
axs.legend(handles=legend_handles, fontsize=14, loc='best')
axs.set_ylabel('M [$M_{\oplus}$]', fontsize=25, labelpad=20)
axs.set_title("Different viscosities$", fontsize=15, pad=20)
plt.savefig("figures/opacities/alpha_params.png", bbox_inches='tight')
plt.show()